# 🤖 Qwen3 LoRA SFT + Embedding Retrieval Few-shot (v2)

## 파이프라인
```
dialogue
    ↓
embedding search (train 12457개와 유사도 계산 → top3 반환)
    ↓
few-shot prompt 생성 (system + 예시3개 + 타겟)
    ↓
Qwen3 생성 → 후처리 (20단어 초과 자르기)
```

## 수정사항
| 항목 | 기존 | 수정 |
|------|------|------|
| MAX_SEQ_LENGTH | 2048 | **1024** |
| PER_DEVICE_BATCH | 2 | **1** |
| GRAD_ACCUM | 16 | **8** |
| LORA_R / ALPHA | 32 / 32 | **16 / 16** |
| MAX_NEW_TOKENS | 192 | **128** |
| Triton | 활성화 | **비활성화** |
| 추론 방식 | zero-shot | **Embedding Retrieval Few-shot** |
| 후처리 | 없음 | **20단어 초과 자르기** |


## ⚙️ 1. 패키지 설치

In [ ]:
import os
os.environ["UNSLOTH_DISABLE_TRITON"] = "1"  # C 컴파일러 없는 환경용
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

!pip install unsloth peft trl datasets rouge python-mecab-ko sentence-transformers scikit-learn -q
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev -q


## ⚙️ 2. Import & 시드 고정

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rouge import Rouge
from mecab import MeCab
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

os.environ["UNSLOTH_DISABLE_TRITON"] = "1"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    used  = torch.cuda.memory_allocated() / 1e9
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {total:.1f}GB 전체 | {total - used:.1f}GB 여유")


## ⚙️ 3. 경로 & 하이퍼파라미터

In [ ]:
# ============================================================
# ⭐ RTX 3090 (24GB) 안정 설정
# ============================================================
DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"
os.makedirs(RESULT_PATH, exist_ok=True)

EXP_NAME        = "qwen3_8b_retrieval_v2"
QWEN_MODEL_NAME = "unsloth/Qwen3-8B"   # ⭐ 동작 확인된 모델명
MAX_SEQ_LENGTH  = 1024    # 2048 → 1024

LORA_R       = 16         # 32 → 16
LORA_ALPHA   = 16         # 32 → 16
LORA_DROPOUT = 0.0

LEARNING_RATE    = 2e-4
EPOCHS           = 3
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
PER_DEVICE_BATCH = 1      # 2 → 1
GRAD_ACCUM       = 8      # 16 → 8

MAX_NEW_TOKENS = 128      # 192 → 128

print(f"Effective batch size : {PER_DEVICE_BATCH * GRAD_ACCUM}")


## ⚙️ 4. 데이터 로드

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print(f"Train : {len(train_df):,}개")
print(f"Dev   : {len(dev_df):,}개")
print(f"Test  : {len(test_df):,}개")


## 🔍 5. Embedding 모델 로드 (CPU)

GPU 메모리 문제 방지를 위해 **CPU**에서 실행


In [ ]:
embed_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"   # GPU VRAM 아끼기 위해 CPU 사용
)
print("Embedding 모델 로드 완료")


## 🔍 6. Train Embedding 구축 (최초 1회만 실행)

In [ ]:
# ✅ 딱 한 번만 실행하면 됨 (약 5~10분)
train_embeddings = embed_model.encode(
    ["passage: " + d for d in train_df["dialogue"]],
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("embedding shape:", train_embeddings.shape)
# 예상 출력: embedding shape: (12457, 768)


## 🔍 7. Retrieval 함수

```
dialogue
    ↓
train 12457개와 similarity 계산
    ↓
가장 비슷한 3개 반환
```


In [ ]:
def retrieve_fewshots(dialogue, k=3):
    """
    Embedding 유사도로 train에서 few-shot k개 선택
    - e5 모델은 query: / passage: prefix 필수
    """
    query_vec = embed_model.encode(
        ["query: " + dialogue],
        convert_to_numpy=True
    )
    sims    = cosine_similarity(query_vec, train_embeddings).flatten()
    top_idx = sims.argsort()[::-1][:k]
    return train_df.iloc[top_idx]


# ✅ 테스트
print("[Retrieval 테스트]")
shots = retrieve_fewshots(dev_df.iloc[0]["dialogue"], k=3)
for i, (_, row) in enumerate(shots.iterrows()):
    print(f"  [{i+1}] 요약: {row['summary']}")


## 📝 8. System Prompt & Prompt 생성 함수

In [ ]:
SYSTEM_PROMPT = """당신은 한국어 대화 요약 전문가입니다.

규칙:
1. 반드시 1문장으로 작성합니다.
2. 20단어 이하로 작성합니다.
3. 대화의 가장 핵심 사건 1개만 포함합니다.
4. 불필요한 세부 설명은 포함하지 않습니다.
5. #Person1#, #Person2# 태그는 절대 변경하지 않습니다.

요약 형식 예시:
- #Person1#이 #Person2#에게 ~을 제안한다.
- #Person1#은 #Person2#에게 ~을 요청한다.
- #Person1#과 #Person2#는 ~에 대해 이야기한다."""


def build_qwen_prompt(dialogue):
    """
    Retrieval Few-shot 프롬프트 구성

    실제 구조:
    system: 요약 전문가
    user: (train dialogue 1)
    assistant: (train summary 1)
    user: (train dialogue 2)
    assistant: (train summary 2)
    user: (train dialogue 3)
    assistant: (train summary 3)
    user: (test dialogue)  ← 타겟
    """
    fewshots = retrieve_fewshots(dialogue, k=3)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for _, row in fewshots.iterrows():
        messages.append({"role": "user",      "content": row["dialogue"]})
        messages.append({"role": "assistant", "content": row["summary"]})

    messages.append({"role": "user", "content": dialogue})

    return messages


# 프롬프트 예시 확인
print("[프롬프트 구조 확인]")
sample_prompt = build_qwen_prompt(dev_df.iloc[0]["dialogue"])
for msg in sample_prompt:
    role = msg["role"]
    content_preview = msg["content"][:60].replace("\n", " ")
    print(f"  [{role}] {content_preview}...")


## 📝 9. 학습용 프롬프트 함수

> **학습 시**: system + user(대화) + assistant(요약) → Response-Only Loss  
> **추론 시**: system + few-shot×3 + user(대화) → Retrieval Few-shot


In [ ]:
def create_messages_train(dialogue, summary):
    """학습용: few-shot 없이 system + user + assistant"""
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": dialogue},
        {"role": "assistant", "content": str(summary)},
    ]


## 🤖 10. Qwen3 모델 로드 (4-bit 양자화)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

torch.cuda.empty_cache()

total = torch.cuda.get_device_properties(0).total_memory / 1e9
used  = torch.cuda.memory_allocated() / 1e9
print(f"✅ 모델 로드 완료: {QWEN_MODEL_NAME}")
print(f"   VRAM: {used:.1f}GB 사용 / {total:.1f}GB 전체 | {total-used:.1f}GB 여유")


## 🤖 11. LoRA 어댑터 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 🤖 12. Response-Only DataCollator

In [ ]:
@dataclass
class ResponseOnlyCollator:
    """assistant 응답 부분만 loss 계산 (프롬프트 마스킹)"""
    tokenizer: Any
    response_template_ids: List[int] = None
    max_length: int = 1024

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(
                features, padding=True,
                max_length=self.max_length, return_tensors="pt"
            )
        else:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts, return_tensors="pt", padding=True,
                truncation=True, max_length=self.max_length
            )
        labels = batch["input_ids"].clone()
        for i in range(len(labels)):
            ids = batch["input_ids"][i].tolist()
            pos = self._find_start(ids)
            if pos >= 0:
                labels[i, :pos] = -100
            labels[i, batch["attention_mask"][i] == 0] = -100
        batch["labels"] = labels
        return batch

    def _find_start(self, ids: List[int]) -> int:
        tmpl = self.response_template_ids
        if not tmpl: return 0
        last = -1
        for i in range(len(ids) - len(tmpl) + 1):
            if ids[i:i+len(tmpl)] == tmpl:
                last = i + len(tmpl)
        return last


resp_tmpl_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
collator      = ResponseOnlyCollator(
    tokenizer=tokenizer,
    response_template_ids=resp_tmpl_ids,
    max_length=MAX_SEQ_LENGTH
)
print(f"Response template IDs: {resp_tmpl_ids}")


## 🤖 13. 학습 데이터 준비

In [ ]:
def formatting_func(examples):
    texts = []
    for d, s in zip(examples["dialogue"], examples["summary"]):
        msg  = create_messages_train(d, s)
        text = tokenizer.apply_chat_template(
            msg, tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}


train_dataset = Dataset.from_pandas(train_df[["dialogue","summary"]]).map(formatting_func, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df[["dialogue","summary"]]).map(formatting_func,   batched=True)

print(f"학습: {len(train_dataset):,}개 | 검증: {len(dev_dataset):,}개")


## 🏋️ 14. SFT 학습 (약 1.5~2시간)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

OUTPUT_DIR = f"/data/ephemeral/home/code/outputs/{EXP_NAME}"

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="wandb",
    optim="adamw_8bit",
    max_grad_norm=1.0,
    run_name=EXP_NAME,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_dataset, eval_dataset=dev_dataset,
    args=sft_config, data_collator=collator,
)

print(f"학습 시작: effective_batch={PER_DEVICE_BATCH*GRAD_ACCUM}, BF16={is_bfloat16_supported()}")
trainer_stats = trainer.train()
print(f"\n✅ 학습 완료! loss={trainer_stats.metrics['train_loss']:.4f}")


In [ ]:
# LoRA 저장
LORA_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"✅ LoRA 저장: {LORA_PATH}")


## 🚀 15. 추론 함수 (Retrieval Few-shot)

학습 완료 후 실행


In [ ]:
# 추론 모드 전환
FastLanguageModel.for_inference(model)


def qwen_summarize(dialogue):
    """Qwen3 추론 - Embedding Retrieval Few-shot"""
    messages = build_qwen_prompt(dialogue)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.2,
        top_p=0.9,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # assistant 응답 부분만 추출
    result = result.split("assistant")[-1].strip()

    # 20단어 초과 후처리 (ROUGE precision 보호)
    words = result.split()
    if len(words) > 20:
        result = " ".join(words[:20])

    return result


# 추론 예시 확인
print("[추론 예시 (dev 3개)]")
for i in range(3):
    pred = qwen_summarize(dev_df.iloc[i]["dialogue"])
    gold = dev_df.iloc[i]["summary"]
    print(f"\n[{i+1}] 정답: {gold}")
    print(f"     예측: {pred}")


## 📊 16. Dev Set ROUGE 평가 (MeCab 기반)

In [ ]:
print(f"Dev 추론 시작 ({len(dev_df)}개)...")
dev_preds = []
for _, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
    pred = qwen_summarize(row["dialogue"])
    dev_preds.append(pred)

rouge = Rouge()
m     = MeCab()

def tok(text):
    tokens = [t for t, _ in m.pos(str(text)) if t.strip()]
    return " ".join(tokens) if tokens else str(text)

golds     = [str(s).strip() for s in dev_df["summary"]]
preds_tok = [tok(p) for p in dev_preds]
golds_tok = [tok(g) for g in golds]

scores = rouge.get_scores(preds_tok, golds_tok, avg=True)
r1  = scores["rouge-1"]["f"]
r2  = scores["rouge-2"]["f"]
rl  = scores["rouge-l"]["f"]
avg = (r1 + r2 + rl) / 3

print(f"\n{'='*50}")
print(f"[{EXP_NAME}] Dev ROUGE (MeCab 기반)")
print(f"  ROUGE-1 : {r1:.4f}")
print(f"  ROUGE-2 : {r2:.4f}")
print(f"  ROUGE-L : {rl:.4f}")
print(f"  AVG     : {avg:.4f}  ← 리더보드 점수와 유사")
print(f"{'='*50}")
print(f"예측 평균 길이: {np.mean([len(p.split()) for p in dev_preds]):.1f}단어")
print(f"#Person 포함율: {sum(1 for p in dev_preds if '#Person' in p)/len(dev_preds):.1%}")


## 🏁 17. Test 추론 & 제출 파일 생성

In [ ]:
print(f"Test 추론 시작 ({len(test_df)}개)...")

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    summary = qwen_summarize(row["dialogue"])
    predictions.append(summary)

# 제출 파일 생성
submission = pd.DataFrame({
    "fname"  : test_df["fname"],
    "summary": predictions,
})

save_path = os.path.join(RESULT_PATH, f"submission_{EXP_NAME}.csv")
submission.to_csv(save_path, index=False)

print(f"\n✅ 제출 파일 저장: {save_path}")
print(f"총 {len(submission)}개 | 고유 요약: {submission['summary'].nunique()}개")
print(f"평균 길이: {submission['summary'].str.split().str.len().mean():.1f}단어")
print(f"#Person 포함: {submission['summary'].str.contains('#Person').mean():.1%}")
print(f"\n[미리보기]")
print(submission.head(10).to_string())


## 18. 새 세션에서 LoRA 재로드 (참고)

학습 완료 후 새 세션에서 추론만 할 때:

```python
import os
os.environ["UNSLOTH_DISABLE_TRITON"] = "1"

from unsloth import FastLanguageModel

LORA_PATH = "/data/ephemeral/home/code/outputs/qwen3_8b_retrieval_v2/lora_adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
# → Embedding 인덱스 구축 (Cell 6) 후 qwen_summarize() 사용
```
